# Lab 05: SVD, Pseudoinverse, Trace & Determinant

**Reference:** Goodfellow et al. *Deep Learning*, Chapter 2, Sections 2.8–2.11

---

## What this lab covers

| Section | Topic | Core idea |
|---------|-------|-----------|
| 2.8 | **Singular Value Decomposition** | *Every* matrix = rotate, scale, rotate |
| 2.9 | **Moore–Penrose Pseudoinverse** | Best substitute for $A^{-1}$ when it does not exist |
| 2.10 | **Trace** | Sum of diagonal / sum of eigenvalues — enables the "trace trick" |
| 2.11 | **Determinant** | Product of eigenvalues — measures volume scaling |

These four topics form a tightly connected web. SVD is arguably the most important matrix factorization in all of machine learning — it underlies PCA, recommender systems, least-squares solutions, and much more. The pseudoinverse is *defined* through SVD. Trace and determinant connect eigenvalues to scalar summaries of a matrix that appear everywhere in loss functions, probability distributions, and optimization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg
import time
import math
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
print('Setup complete.')

---

## Part 1: Singular Value Decomposition (Section 2.8)

### The Big Idea

Eigendecomposition ($A = Q\Lambda Q^{-1}$) only works for *square* matrices, and it might not even exist (defective matrices). SVD has **no such limitation**.

> **Theorem (SVD).** Every real matrix $A \in \mathbb{R}^{m \times n}$ can be factored as:
> $$A = U \Sigma V^\top$$
> where:
> - $U \in \mathbb{R}^{m \times m}$ is orthogonal (columns are the **left singular vectors**),
> - $\Sigma \in \mathbb{R}^{m \times n}$ is diagonal with non-negative entries $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$ (the **singular values**),
> - $V \in \mathbb{R}^{n \times n}$ is orthogonal (columns are the **right singular vectors**).

Read that first line again: **every** matrix. Tall, wide, square, rank-deficient — it does not matter. SVD always exists.

### Geometric Meaning: Rotate $\to$ Scale $\to$ Rotate

This is the most important intuition:

$$\underbrace{A}_{\text{any linear map}} = \underbrace{U}_{\text{rotate/reflect}} \;\underbrace{\Sigma}_{\text{scale axes}} \;\underbrace{V^\top}_{\text{rotate/reflect}}$$

Every linear transformation, no matter how complicated, decomposes into three simple steps:
1. **$V^\top$:** Rotate (or reflect) the input space to align with the "natural axes" of the transformation.
2. **$\Sigma$:** Scale each axis independently by $\sigma_i$.
3. **$U$:** Rotate (or reflect) the result into the output space.

This is profound. It means the *only* interesting thing a matrix does is stretch space along certain directions. Everything else is just rotation.

In [ ]:
# Demo: Geometric meaning of SVD -- watch the unit circle transform step by step

theta = np.linspace(0, 2 * np.pi, 300)
unit_circle = np.array([np.cos(theta), np.sin(theta)])  # shape (2, 300)

# A 2x2 matrix -- any matrix works
A = np.array([[3.0, 1.0],
              [1.0, 2.0]])

U, s, Vt = np.linalg.svd(A)
Sigma = np.diag(s)

# Apply each step separately so we can visualize
step1 = Vt @ unit_circle           # after V^T
step2 = Sigma @ step1              # after Sigma
step3 = U @ step2                  # after U  (= A @ unit_circle)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

titles = [
    'Original unit circle',
    r'Step 1: After $V^\top$ (rotation)',
    r'Step 2: After $\Sigma$ (scaling)',
    r'Step 3: After $U$ (rotation) $= A\mathbf{x}$'
]

for ax, pts, title in zip(axes, [unit_circle, step1, step2, step3], titles):
    ax.plot(pts[0], pts[1], 'b-', linewidth=2)
    # Mark where (1,0) ends up -- helps track the rotation
    ax.plot(pts[0, 0], pts[1, 0], 'ro', markersize=8)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=9)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)

plt.suptitle(r'SVD Geometric Interpretation: $A = U \Sigma V^\top$', fontsize=12)
plt.tight_layout()
plt.show()

print(f'A =\n{A}')
print(f'\nSingular values: sigma_1 = {s[0]:.4f}, sigma_2 = {s[1]:.4f}')
print(f'  These are the semi-axis lengths of the output ellipse.')
print(f'\nU (left singular vectors -- output rotation) =\n{U}')
print(f'\nV (right singular vectors -- input rotation) =\n{Vt.T}')

**What just happened:** The unit circle (all vectors of length 1) got mapped to an ellipse. The singular values $\sigma_1, \sigma_2$ are the semi-axis lengths of that ellipse. The columns of $V$ point in the directions that get mapped to the axes of the ellipse. The columns of $U$ point along the axes of the output ellipse.

### SVD and Eigendecomposition: The Connection

Where do singular values come from? Consider $A^\top A$:

$$A^\top A = (U \Sigma V^\top)^\top (U \Sigma V^\top) = V \Sigma^\top \Sigma V^\top = V \,\text{diag}(\sigma_i^2)\, V^\top$$

This is an eigendecomposition of $A^\top A$. So:
- **Singular values** $\sigma_i = \sqrt{\lambda_i(A^\top A)}$ — square roots of eigenvalues of $A^\top A$.
- **Right singular vectors** $V$ = eigenvectors of $A^\top A$.
- **Left singular vectors** $U$ = eigenvectors of $A A^\top$.

This also explains why SVD always exists: $A^\top A$ is symmetric positive semidefinite, so its eigendecomposition always exists with non-negative eigenvalues.

### Exercise 1 (Basic): SVD Verification

Compute the SVD of a **rectangular** matrix (not square!), reconstruct $A$ from $U, \Sigma, V^\top$, and verify that $U$ and $V$ are orthogonal ($U^\top U = I$ and $V^\top V = I$).

In [ ]:
# Exercise 1: SVD verification
A = np.random.randn(4, 3)

# 1. Compute the full SVD of A using np.linalg.svd with full_matrices=True
U, s, Vt = np.linalg.svd(A, full_matrices=True)

# 2. Build the full Sigma matrix (shape m x n) from the singular values vector s
#    Note: np.linalg.svd returns s as a 1-D array of min(m,n) singular values,
#    but Sigma must be m x n with those values on the diagonal.
m_A, n_A = A.shape
Sigma_full = np.zeros((m_A, n_A))
np.fill_diagonal(Sigma_full, s)

# 3. Reconstruct A_reconstructed = U @ Sigma_full @ Vt
A_reconstructed = U @ Sigma_full @ Vt

# 4. Check orthogonality: U^T U should be I(m), V^T V should be I(n)
V = Vt.T
U_ortho_error = U.T @ U - np.eye(U.shape[0])
V_ortho_error = V.T @ V - np.eye(V.shape[0])
recon_error = np.max(np.abs(A_reconstructed - A))

print(f'A shape: {A.shape}')
print(f'U shape: {U.shape}, s shape: {s.shape}, Vt shape: {Vt.shape}')
print(f'Sigma shape: {Sigma_full.shape} (m x n, with min(m,n) singular values on diagonal)')
print(f'\nReconstruction error (max |A - U Sigma Vt|): {recon_error:.2e}')
print(f'U orthogonality error (max |U^T U - I|): {np.max(np.abs(U_ortho_error)):.2e}')
print(f'V orthogonality error (max |V^T V - I|): {np.max(np.abs(V_ortho_error)):.2e}')

In [ ]:
# Verification cell for Exercise 1
assert U.shape == (4, 4), f'U should be 4x4, got {U.shape}'
assert s.shape == (3,), f's should have 3 singular values, got {s.shape}'
assert Vt.shape == (3, 3), f'Vt should be 3x3, got {Vt.shape}'
assert np.allclose(A_reconstructed, A, atol=1e-10), 'Reconstruction A = U Sigma Vt failed'
assert np.allclose(U_ortho_error, np.zeros((4,4)), atol=1e-10), 'U is not orthogonal'
assert np.allclose(V_ortho_error, np.zeros((3,3)), atol=1e-10), 'V is not orthogonal'
print('Exercise 1 PASSED')
print(f'  Reconstruction error: {recon_error:.2e}')
print(f'  U orthogonality error (max |U^T U - I|): {np.max(np.abs(U_ortho_error)):.2e}')
print(f'  V orthogonality error (max |V^T V - I|): {np.max(np.abs(V_ortho_error)):.2e}')

### Exercise 2 (Basic): SVD–Eigendecomposition Relationship

We claimed that the singular values of $A$ are $\sigma_i = \sqrt{\lambda_i(A^\top A)}$ and the right singular vectors are the eigenvectors of $A^\top A$. Let us verify this directly.

In [ ]:
# Exercise 2: SVD-eigendecomposition relationship
A = np.random.randn(5, 3)

# 1. Compute SVD of A (economy/thin SVD: full_matrices=False)
U, s, Vt = np.linalg.svd(A, full_matrices=False)

# 2. Compute eigendecomposition of A^T A using np.linalg.eigh (symmetric matrix)
#    eigh is for symmetric/Hermitian matrices -- returns real eigenvalues
#    Note: eigh returns eigenvalues in ascending order; we need descending
AtA = A.T @ A
eigvals, eigvecs = np.linalg.eigh(AtA)

# Sort descending
idx = np.argsort(eigvals)[::-1]
eigvals_sorted = eigvals[idx]
eigvecs_sorted = eigvecs[:, idx]

# 3. Verify: singular values s == sqrt(eigenvalues of A^T A)
sv_from_eig = np.sqrt(np.maximum(eigvals_sorted, 0.0))

# Compare singular values
singular_val_match = s - sv_from_eig

# 4. Compare V columns (allow sign flips -- eigenvectors are unique only up to sign)
V = Vt.T
vec_alignment = np.array([
    abs(np.dot(V[:, j], eigvecs_sorted[:, j]))
    for j in range(V.shape[1])
])

print('Singular values from SVD:              ', s)
print('Singular values from sqrt(eig(A^T A)): ', sv_from_eig)
print('Difference:                             ', singular_val_match)
print(f'\nV column alignments (|dot product|, should all be 1.0): {vec_alignment}')

In [ ]:
# Verification cell for Exercise 2
assert np.allclose(singular_val_match, np.zeros_like(s), atol=1e-10), \
    f'Singular values do not match sqrt(eigenvalues of A^T A). Diff: {singular_val_match}'
assert np.allclose(vec_alignment, np.ones(V.shape[1]), atol=1e-10), \
    f'V columns do not match eigenvectors of A^T A. Alignments: {vec_alignment}'
print('Exercise 2 PASSED')
print(f'  Singular values from SVD:         {s}')
print(f'  Singular values from sqrt(eig):   {sv_from_eig}')
print(f'  Max difference:                   {np.max(np.abs(singular_val_match)):.2e}')
print(f'  V column alignments (should be 1): {vec_alignment}')

### Low-Rank Approximation: The Eckart–Young Theorem

This is one of SVD's most powerful applications. Suppose you want to approximate a matrix $A$ with a simpler (lower-rank) matrix. Which rank-$k$ matrix is closest?

> **Theorem (Eckart–Young).** The best rank-$k$ approximation to $A$ in the Frobenius norm is the truncated SVD:
> $$A_k = U_k \Sigma_k V_k^\top = \sum_{i=1}^{k} \sigma_i \, u_i \, v_i^\top$$
> where $U_k, \Sigma_k, V_k$ keep only the top $k$ singular values/vectors.
>
> The approximation error is:
> $$\|A - A_k\|_F = \sqrt{\sum_{i=k+1}^{r} \sigma_i^2}$$

**Why this matters for ML:**
- **PCA** is exactly truncated SVD of the centered data matrix.
- **Latent Semantic Analysis (LSA)** uses truncated SVD of the term-document matrix.
- **Recommender systems** (Netflix prize!) use SVD-based matrix factorization.
- **Image compression**: store $k(m + n + 1)$ numbers instead of $mn$.

### Exercise 3 (Intermediate): Low-Rank Approximation (Eckart–Young Theorem)

Verify the theorem: compute rank-$k$ approximations for $k = 1, 2, \ldots, \min(m, n)$ and compare actual vs. theoretical error.

In [ ]:
# Exercise 3: Low-rank approximation and Eckart-Young theorem
A = np.random.randn(8, 6)

def truncated_svd(U, s, Vt, k):
    """Return the best rank-k approximation of A = U diag(s) Vt."""
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

U, s, Vt = np.linalg.svd(A, full_matrices=False)

ks = list(range(1, min(A.shape) + 1))
actual_errors = []
theoretical_errors = []

for k in ks:
    A_k = truncated_svd(U, s, Vt, k)
    actual_err = np.linalg.norm(A - A_k, 'fro')
    theoretical_err = np.sqrt(np.sum(s[k:] ** 2))
    actual_errors.append(actual_err)
    theoretical_errors.append(theoretical_err)

plt.figure(figsize=(8, 4))
plt.plot(ks, actual_errors, 'bo-', label=r'Actual $\|A - A_k\|_F$')
plt.plot(ks, theoretical_errors, 'r--', marker='x',
         label=r'Theoretical $\sqrt{\sum_{i>k}\sigma_i^2}$')
plt.xlabel('Rank k')
plt.ylabel('Frobenius Error')
plt.title('Eckart-Young Theorem: Low-Rank Approximation Error')
plt.legend()
plt.tight_layout()
plt.show()

print('Singular values:', s.round(4))
for k, ae, te in zip(ks, actual_errors, theoretical_errors):
    print(f'  k={k}: actual error={ae:.6f}, theoretical={te:.6f}, match={np.isclose(ae, te)}')

In [ ]:
# Verification cell for Exercise 3
for i, k in enumerate(ks):
    assert np.isclose(actual_errors[i], theoretical_errors[i], atol=1e-10), \
        f'Eckart-Young failed at k={k}: actual={actual_errors[i]:.6f}, theoretical={theoretical_errors[i]:.6f}'
print('Exercise 3 PASSED: Eckart-Young theorem verified for all k')
for k, ae, te in zip(ks, actual_errors, theoretical_errors):
    print(f'  k={k}: actual={ae:.6f}, theoretical={te:.6f}, diff={abs(ae-te):.2e}')

### Exercise 4 (Intermediate): Image Compression with SVD

A concrete application of low-rank approximation: compress an image.

An $m \times n$ image needs $mn$ numbers to store. A rank-$k$ SVD approximation stores:
- $U_k$: $m \times k$ numbers
- $\Sigma_k$: $k$ numbers
- $V_k^\top$: $k \times n$ numbers

Total: $k(m + n + 1)$ numbers. Compression ratio: $\frac{mn}{k(m + n + 1)}$.

In [ ]:
# Exercise 4: Image compression with SVD
# Create a synthetic grayscale image with structure (not random noise)
m, n = 64, 64
xx, yy = np.meshgrid(np.linspace(0, 4*np.pi, n), np.linspace(0, 4*np.pi, m))
image = (np.sin(xx) * np.cos(yy) + np.sin(2*xx) * np.cos(3*yy) +
         0.3 * np.random.randn(m, n))
image = (image - image.min()) / (image.max() - image.min())  # normalize to [0,1]

# 1. Compute the full SVD of the image
U, s, Vt = np.linalg.svd(image, full_matrices=False)

ranks = [1, 5, 10, 20]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(image, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original\n(64x64 = 4096 values)', fontsize=9)
axes[0].axis('off')

for ax, k in zip(axes[1:], ranks):
    # 2a. Compute the rank-k approximation
    approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    # 2b. Clip values to [0, 1]
    approx_clipped = np.clip(approx, 0, 1)
    # 2c. Compute compression ratio
    storage = k * (m + n + 1)
    compression_ratio = (m * n) / storage
    # 2d. Compute PSNR (peak signal-to-noise ratio)
    mse = np.mean((image - approx_clipped) ** 2)
    psnr = 20 * np.log10(1.0 / np.sqrt(mse)) if mse > 0 else float('inf')

    ax.imshow(approx_clipped, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Rank {k}\n{storage} values, CR={compression_ratio:.1f}x\nPSNR={psnr:.1f}dB', fontsize=9)
    ax.axis('off')

plt.suptitle('SVD Image Compression: Keep Top-k Singular Values', fontsize=12)
plt.tight_layout()
plt.show()

# Show how energy concentrates in the top singular values
energy_cumulative = np.cumsum(s**2) / np.sum(s**2) * 100
print(f'Top 10 singular values: {s[:10].round(2)}')
print(f'Cumulative energy: rank-5 captures {energy_cumulative[4]:.1f}%, '
      f'rank-10 captures {energy_cumulative[9]:.1f}%, '
      f'rank-20 captures {energy_cumulative[19]:.1f}%')

**Key insight:** Real-world data (images, text, user-item matrices) typically has **rapidly decaying singular values** — most of the "energy" (information) lives in the first few singular values. This is why low-rank approximations work so well in practice, and why PCA is so effective for dimensionality reduction.

---

## Part 2: Moore–Penrose Pseudoinverse (Section 2.9)

### The Problem

The matrix inverse $A^{-1}$ only exists when $A$ is square and invertible (full rank). But in ML, we constantly encounter:
- **Overdetermined systems** ($m > n$): more equations than unknowns (e.g., fitting a model to 1000 data points with 10 parameters). No exact solution exists.
- **Underdetermined systems** ($m < n$): more unknowns than equations. Infinitely many solutions exist.
- **Rank-deficient matrices**: even if square, some directions are collapsed.

The **Moore–Penrose pseudoinverse** $A^+$ handles all of these cases. It is the "best substitute" for $A^{-1}$.

### Definition via SVD

Given the SVD $A = U \Sigma V^\top$, the pseudoinverse is:
$$A^+ = V \Sigma^+ U^\top$$

where $\Sigma^+$ is obtained by **reciprocating every non-zero singular value** (and leaving zeros as zeros):
$$\Sigma^+_{ii} = \begin{cases} 1/\sigma_i & \text{if } \sigma_i > 0 \\ 0 & \text{if } \sigma_i = 0 \end{cases}$$

The intuition is simple: SVD says $A$ = rotate, scale, rotate. The pseudoinverse = un-rotate, un-scale (where possible), un-rotate. Directions that were scaled to zero cannot be recovered, so those stay zero.

### Two Key Applications

| System | Shape | $A^+ b$ gives | Why |
|--------|-------|---------------|-----|
| Overdetermined ($m > n$) | tall | **Least-squares** solution: minimizes $\|Ax - b\|^2$ | Best fit when no exact solution exists |
| Underdetermined ($m < n$) | wide | **Minimum-norm** solution: smallest $\|x\|$ among all solutions | Simplest solution when many exist |

### ML Connection

Linear regression's closed-form solution **is** the pseudoinverse:
$$\hat{\theta} = X^+ y = (X^\top X)^{-1} X^\top y$$
(The right-hand form is valid when $X^\top X$ is invertible; the pseudoinverse works even when it is not.)

### Exercise 5 (Basic): Implement the Pseudoinverse via SVD

Implement $A^+ = V \Sigma^+ U^\top$ from scratch and compare to `np.linalg.pinv`.

In [ ]:
# Exercise 5: Pseudoinverse via SVD

def pseudoinverse(A, tol=None):
    """Compute Moore-Penrose pseudoinverse via SVD.
    
    Algorithm:
    1. Compute thin SVD: A = U diag(s) Vt
    2. Reciprocate non-zero singular values (using tolerance to handle near-zero)
    3. Return Vt.T @ diag(s_inv) @ U.T
    """
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    m_loc, n_loc = A.shape
    if tol is None:
        eps = np.finfo(float).eps
        tol = max(m_loc, n_loc) * eps * s[0]
    s_inv = np.where(s > tol, 1.0 / s, 0.0)
    return Vt.T @ np.diag(s_inv) @ U.T

# Test on tall (overdetermined) matrix
A_tall = np.random.randn(6, 3)
A_pinv_custom = pseudoinverse(A_tall)
A_pinv_numpy = np.linalg.pinv(A_tall)
tall_error = np.max(np.abs(A_pinv_custom - A_pinv_numpy))

# Test on wide (underdetermined) matrix
A_wide = np.random.randn(3, 6)
A_wide_pinv_custom = pseudoinverse(A_wide)
A_wide_pinv_numpy = np.linalg.pinv(A_wide)
wide_error = np.max(np.abs(A_wide_pinv_custom - A_wide_pinv_numpy))

# Test on rank-deficient matrix (4x4 but only rank 2)
A_rank_def = np.random.randn(4, 2) @ np.random.randn(2, 4)  # rank 2 matrix (4x4)
A_rd_pinv_custom = pseudoinverse(A_rank_def)
A_rd_pinv_numpy = np.linalg.pinv(A_rank_def)
rank_def_error = np.max(np.abs(A_rd_pinv_custom - A_rd_pinv_numpy))

print(f'Tall matrix (6x3) -- pseudoinverse shape: {A_pinv_custom.shape}')
print(f'  Max error vs np.linalg.pinv: {tall_error:.2e}')
print(f'\nWide matrix (3x6) -- pseudoinverse shape: {A_wide_pinv_custom.shape}')
print(f'  Max error vs np.linalg.pinv: {wide_error:.2e}')
print(f'\nRank-deficient matrix (4x4, rank 2) -- pseudoinverse shape: {A_rd_pinv_custom.shape}')
print(f'  Max error vs np.linalg.pinv: {rank_def_error:.2e}')

In [ ]:
# Verification cell for Exercise 5
assert np.allclose(A_pinv_custom, A_pinv_numpy, atol=1e-10), \
    f'Pseudoinverse of tall matrix wrong. Max error: {tall_error:.2e}'
assert np.allclose(A_wide_pinv_custom, A_wide_pinv_numpy, atol=1e-10), \
    f'Pseudoinverse of wide matrix wrong. Max error: {wide_error:.2e}'
assert np.allclose(A_rd_pinv_custom, A_rd_pinv_numpy, atol=1e-10), \
    f'Pseudoinverse of rank-deficient matrix wrong. Max error: {rank_def_error:.2e}'
print('Exercise 5 PASSED')
print(f'  Tall matrix max error:           {tall_error:.2e}')
print(f'  Wide matrix max error:           {wide_error:.2e}')
print(f'  Rank-deficient matrix max error: {rank_def_error:.2e}')

### Exercise 6 (Intermediate): Least Squares via Pseudoinverse

For an overdetermined system $Ax = b$ (more equations than unknowns), there is generally no exact solution. The pseudoinverse gives the **least-squares** solution:
$$x^* = A^+ b = \arg\min_x \|Ax - b\|_2^2$$

This is exactly what linear regression does: given data matrix $X$ and targets $y$, find $\theta$ minimizing $\|X\theta - y\|^2$. The answer is $\theta = X^+ y$.

In [ ]:
# Exercise 6: Least squares via pseudoinverse
# Overdetermined system: 10 equations, 3 unknowns
m, n = 10, 3
A = np.random.randn(m, n)
x_true = np.array([1.0, -2.0, 3.0])
b = A @ x_true + 0.1 * np.random.randn(m)  # add noise -> no exact solution

# Three equivalent methods:

# Method 1: Pseudoinverse: x = A^+ b
x_pinv = pseudoinverse(A) @ b

# Method 2: np.linalg.lstsq (uses SVD internally)
x_lstsq, residuals, rank, sv = np.linalg.lstsq(A, b, rcond=None)

# Method 3: Normal equations: x = (A^T A)^{-1} A^T b
x_normal = np.linalg.solve(A.T @ A, A.T @ b)

# Compute residuals for all methods
residual_pinv = np.linalg.norm(A @ x_pinv - b)
residual_lstsq = np.linalg.norm(A @ x_lstsq - b)
residual_normal = np.linalg.norm(A @ x_normal - b)

print('All three methods should give the same answer:')
print(f'  x_pinv   = {x_pinv}')
print(f'  x_lstsq  = {x_lstsq}')
print(f'  x_normal = {x_normal}')
print(f'  x_true   = {x_true}  (would recover exactly if no noise)')
print(f'\nResiduals (all should match):')
print(f'  ||Ax_pinv - b||   = {residual_pinv:.6f}')
print(f'  ||Ax_lstsq - b||  = {residual_lstsq:.6f}')
print(f'  ||Ax_normal - b|| = {residual_normal:.6f}')

In [ ]:
# Verification cell for Exercise 6
assert np.allclose(x_pinv, x_lstsq, atol=1e-8), \
    f'Pseudoinverse and lstsq solutions differ. Max diff: {np.max(np.abs(x_pinv - x_lstsq)):.2e}'
assert np.allclose(x_pinv, x_normal, atol=1e-8), \
    f'Pseudoinverse and normal equations solutions differ. Max diff: {np.max(np.abs(x_pinv - x_normal)):.2e}'
assert np.isclose(residual_pinv, residual_lstsq, atol=1e-8), \
    f'Residuals differ: pinv={residual_pinv:.6f}, lstsq={residual_lstsq:.6f}'
print('Exercise 6 PASSED: All three methods give the same least-squares solution')
print(f'  x_pinv:   {x_pinv}')
print(f'  x_lstsq:  {x_lstsq}')
print(f'  x_normal: {x_normal}')
print(f'  Residual ||Ax - b||_2 = {residual_pinv:.6f}')

### Exercise 7 (Intermediate): Minimum Norm Solution

For an underdetermined system $Ax = b$ (more unknowns than equations, infinitely many solutions), the pseudoinverse gives the **minimum-norm** solution:
$$x^* = A^+ b \quad \text{minimizes } \|x\|_2 \text{ subject to } Ax = b$$

Among all solutions, $A^+ b$ picks the one with the smallest Euclidean norm — the "simplest" solution.

In [ ]:
# Exercise 7: Minimum norm solution for underdetermined systems
# Underdetermined: 3 equations, 6 unknowns -> infinitely many solutions
m, n = 3, 6
A = np.random.randn(m, n)
x_particular = np.random.randn(n)
b = A @ x_particular  # exact solution exists

# 1. Compute the minimum norm solution
x_minnorm = pseudoinverse(A) @ b

# 2. Verify it satisfies Ax = b exactly
residual_check = A @ x_minnorm - b

# 3. Generate many other valid solutions by adding null space components
#    Key idea: if x0 solves Ax = b, then x0 + z also solves it whenever Az = 0
#    The null space projector is: P_null = I - A^+ A
A_pinv = np.linalg.pinv(A)
P_null = np.eye(n) - A_pinv @ A  # projection onto null space of A

N = 1000
norms_random = []
for _ in range(N):
    v = np.random.randn(n)
    x_rand = x_minnorm + P_null @ v  # another valid solution
    norms_random.append(np.linalg.norm(x_rand))

norms_random = np.array(norms_random)
norm_minnorm = np.linalg.norm(x_minnorm)

plt.figure(figsize=(8, 4))
plt.hist(norms_random, bins=40, alpha=0.7, label='1000 random valid solutions')
plt.axvline(norm_minnorm, color='r', linewidth=2,
            label=f'Pseudoinverse solution: ||x|| = {norm_minnorm:.3f}')
plt.xlabel(r'$\|x\|_2$')
plt.ylabel('Count')
plt.title('Minimum Norm Property: $A^+b$ has smallest norm among all solutions')
plt.legend()
plt.tight_layout()
plt.show()

print(f'||x_minnorm|| = {norm_minnorm:.6f}')
print(f'Smallest random solution norm = {norms_random.min():.6f}')
print(f'All random solutions have larger norm: {np.all(norms_random >= norm_minnorm - 1e-10)}')

In [ ]:
# Verification cell for Exercise 7
assert np.allclose(residual_check, np.zeros(m), atol=1e-10), \
    f'Minimum norm solution does not satisfy Ax = b. Residual: {residual_check}'
assert np.all(norms_random >= norm_minnorm - 1e-10), \
    f'Some random solution has smaller norm than pseudoinverse solution! Min random norm: {norms_random.min():.6f}, minnorm: {norm_minnorm:.6f}'
print('Exercise 7 PASSED')
print(f'  ||x_minnorm||_2 = {norm_minnorm:.6f}')
print(f'  Min ||x_random||_2 = {norms_random.min():.6f}')
print(f'  Mean ||x_random||_2 = {norms_random.mean():.6f}')
print(f'  Ax = b residual (should be ~0): {np.max(np.abs(residual_check)):.2e}')

---

## Part 3: Trace (Section 2.10)

### Definition

The trace of a square matrix $A \in \mathbb{R}^{n \times n}$ is simply the sum of its diagonal elements:
$$\text{tr}(A) = \sum_{i=1}^{n} A_{ii}$$

### Why Should You Care?

Trace seems like a trivial operation, but it has several deep properties that make it indispensable in ML:

**1. Trace = sum of eigenvalues:**
$$\text{tr}(A) = \sum_{i=1}^{n} \lambda_i$$

**2. Cyclic invariance (the most useful property):**
$$\text{tr}(ABC) = \text{tr}(CAB) = \text{tr}(BCA)$$
Note: you can cycle but NOT freely permute. $\text{tr}(ABC) \neq \text{tr}(ACB)$ in general.

This works even for non-square matrices as long as the dimensions are compatible for the product to be square.

**3. Frobenius norm via trace:**
$$\|A\|_F^2 = \text{tr}(A^\top A) = \sum_{i,j} A_{ij}^2$$

**4. The Trace Trick (gradient identity):**
$$\frac{\partial}{\partial A} \text{tr}(AB) = B^\top$$

This identity is used *constantly* in matrix calculus for ML. When you see a loss function written as $\text{tr}(\text{something})$, the trace trick lets you differentiate it easily.

### Exercise 8 (Basic): Trace Properties

Implement trace from scratch and verify three key properties:
- (a) $\text{tr}(A) = \sum_i \lambda_i$
- (b) $\text{tr}(AB) = \text{tr}(BA)$ (cyclic property)
- (c) $\text{tr}(A^\top B) = \sum_{i,j} A_{ij} B_{ij}$ (Frobenius inner product)

In [ ]:
# Exercise 8: Trace properties

def trace(A):
    """Compute trace of square matrix A without np.trace."""
    return sum(A[i, i] for i in range(min(A.shape)))

# (a) tr(A) = sum of eigenvalues
A = np.random.randn(5, 5)
trace_A = trace(A)
eigenvalues = np.linalg.eigvals(A)
trace_from_eig = np.real(np.sum(eigenvalues))  # eigenvalues may be complex; trace is real

# (b) tr(AB) = tr(BA) for non-square A, B
#     A_rect is 4x6, B_rect is 6x4
#     AB is 4x4, BA is 6x6 -- different sizes, but same trace!
A_rect = np.random.randn(4, 6)
B_rect = np.random.randn(6, 4)
trace_AB = trace(A_rect @ B_rect)  # 4x4 matrix
trace_BA = trace(B_rect @ A_rect)  # 6x6 matrix

# (c) tr(A^T B) = Frobenius inner product = sum_{i,j} A_{ij} B_{ij}
A2 = np.random.randn(4, 5)
B2 = np.random.randn(4, 5)
trace_AtB = trace(A2.T @ B2)       # trace of 5x5 matrix A^T B
frobenius_inner = np.sum(A2 * B2)  # element-wise multiply and sum

print(f'(a) tr(A) = {trace_A:.6f}')
print(f'    sum(eigenvalues) = {trace_from_eig:.6f}')
print(f'    Match: {np.isclose(trace_A, trace_from_eig)}')
print(f'\n(b) tr(AB) = {trace_AB:.6f}  (AB is 4x4)')
print(f'    tr(BA) = {trace_BA:.6f}  (BA is 6x6)')
print(f'    Match: {np.isclose(trace_AB, trace_BA)}')
print(f'\n(c) tr(A^T B) = {trace_AtB:.6f}')
print(f'    sum(A * B) = {frobenius_inner:.6f}')
print(f'    Match: {np.isclose(trace_AtB, frobenius_inner)}')

In [ ]:
# Verification cell for Exercise 8
assert np.isclose(trace_A, np.trace(A), atol=1e-10), \
    f'trace() implementation wrong. Got {trace_A}, expected {np.trace(A)}'
assert np.isclose(trace_A, trace_from_eig, atol=1e-10), \
    f'tr(A) != sum(eigenvalues). Got {trace_A} vs {trace_from_eig}'
assert np.isclose(trace_AB, trace_BA, atol=1e-10), \
    f'tr(AB) != tr(BA). Got {trace_AB} vs {trace_BA}'
assert np.isclose(trace_AtB, frobenius_inner, atol=1e-10), \
    f'tr(A^T B) != Frobenius inner product. Got {trace_AtB} vs {frobenius_inner}'
print('Exercise 8 PASSED: All trace properties verified')
print(f'  (a) tr(A) = {trace_A:.6f}, sum(eigenvalues) = {trace_from_eig:.6f}')
print(f'  (b) tr(AB) = {trace_AB:.6f}, tr(BA) = {trace_BA:.6f}')
print(f'  (c) tr(A^T B) = {trace_AtB:.6f}, Frobenius inner = {frobenius_inner:.6f}')

### Frobenius Norm via Trace

An important corollary: the squared Frobenius norm can be written as a trace:
$$\|A\|_F^2 = \sum_{i,j} A_{ij}^2 = \text{tr}(A^\top A)$$

And connecting back to SVD and singular values:
$$\|A\|_F^2 = \text{tr}(A^\top A) = \sum_i \lambda_i(A^\top A) = \sum_i \sigma_i^2$$

This is why the Eckart-Young error formula involves $\sqrt{\sum_{i > k} \sigma_i^2}$ — it is the Frobenius norm of the residual, which equals the trace of the residual's gram matrix.

### Exercise 9 (Intermediate): Trace Trick in ML -- Gradient Verification

The identity $\frac{\partial}{\partial A} \text{tr}(AB) = B^\top$ is the workhorse of matrix calculus in ML. Let us verify it numerically using finite differences.

**Why this matters:** Many ML loss functions are naturally expressed as traces. For example, the sum of squared errors $\|X\theta - y\|^2 = \text{tr}\big((X\theta - y)^\top (X\theta - y)\big)$. The trace trick lets you differentiate such expressions cleanly.

In [ ]:
# Exercise 9: Trace trick -- verify d/dA tr(AB) = B^T via finite differences
A = np.random.randn(4, 4)
B = np.random.randn(4, 4)

def f(A, B):
    """Compute tr(A @ B)."""
    return np.trace(A @ B)

# 1. Analytical gradient: d/dA tr(AB) = B^T
grad_analytical = B.T

# 2. Numerical gradient via central finite differences
#    For each entry A_{ij}: grad_{ij} = [f(A + eps*E_{ij}) - f(A - eps*E_{ij})] / (2*eps)
eps = 1e-5
grad_numerical = np.zeros_like(A)

for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        A_plus = A.copy()
        A_plus[i, j] += eps
        A_minus = A.copy()
        A_minus[i, j] -= eps
        grad_numerical[i, j] = (f(A_plus, B) - f(A_minus, B)) / (2 * eps)

max_grad_error = np.max(np.abs(grad_analytical - grad_numerical))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
im1 = axes[0].imshow(grad_analytical, cmap='RdBu_r')
axes[0].set_title(r'Analytical: $B^\top$')
plt.colorbar(im1, ax=axes[0])
im2 = axes[1].imshow(grad_numerical, cmap='RdBu_r')
axes[1].set_title('Numerical (finite diff)')
plt.colorbar(im2, ax=axes[1])
im3 = axes[2].imshow(grad_analytical - grad_numerical, cmap='RdBu_r')
axes[2].set_title(f'Error (max={max_grad_error:.2e})')
plt.colorbar(im3, ax=axes[2])
plt.suptitle(r'Verifying: $\frac{\partial}{\partial A} \mathrm{tr}(AB) = B^\top$', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Verification cell for Exercise 9
assert np.allclose(grad_analytical, grad_numerical, atol=1e-7), \
    f'd/dA tr(AB) = B^T verification failed. Max error: {max_grad_error:.2e}'
print('Exercise 9 PASSED: d/dA tr(AB) = B^T verified numerically')
print(f'  Max error between analytical and numerical gradient: {max_grad_error:.2e}')

---

## Part 4: Determinant (Section 2.11)

### Definition and Key Formulas

The determinant of a square matrix $A \in \mathbb{R}^{n \times n}$ is a scalar that encodes fundamental geometric information:

$$\det(A) = \prod_{i=1}^{n} \lambda_i \quad \text{(product of eigenvalues)}$$

Compare with trace: $\text{tr}(A) = \sum_i \lambda_i$ (sum of eigenvalues). Together, trace and determinant give you the two simplest scalar summaries of a matrix's eigenvalues.

### Geometric Meaning: Volume Scaling

The determinant measures how $A$ scales volumes:
- $|\det(A)|$ = factor by which areas (2D) or volumes (3D) or hypervolumes (nD) change.
- $\det(A) = 0$ means the transformation **collapses a dimension** -- the output lives in a lower-dimensional subspace.
- $\det(A) > 0$: orientation is **preserved** (right-hand rule is maintained).
- $\det(A) < 0$: orientation is **flipped** (like a mirror reflection).

### Key Properties

- $\det(AB) = \det(A) \det(B)$
- $\det(A^{-1}) = 1/\det(A)$
- $\det(cA) = c^n \det(A)$ for $A \in \mathbb{R}^{n \times n}$
- $\det(A^\top) = \det(A)$

### ML Connections

Determinants appear frequently in probability and ML:
- **Multivariate Gaussian:** $p(x) = \frac{1}{(2\pi)^{n/2} |\det(\Sigma)|^{1/2}} \exp\Big(-\frac{1}{2}(x-\mu)^\top \Sigma^{-1} (x-\mu)\Big)$. The $\det(\Sigma)$ in the normalizing constant ensures the density integrates to 1.
- **Normalizing flows:** Use the change-of-variables formula: $p_Y(y) = p_X(f^{-1}(y)) \cdot |\det(J_{f^{-1}}(y))|$. The Jacobian determinant accounts for how the transformation stretches/compresses probability density.
- **Log-likelihood of Gaussian mixtures** involves $\log \det(\Sigma_k)$ for each component.

### Exercise 10 (Basic): Determinant Properties

In [ ]:
# Exercise 10: Verify determinant properties
np.random.seed(7)
A = np.random.randn(4, 4)
B = np.random.randn(4, 4)
n = 4
c = 3.0

# Compute all determinants needed
det_A = np.linalg.det(A)
det_B = np.linalg.det(B)

# (a) det(AB) = det(A) * det(B)
det_AB = np.linalg.det(A @ B)
det_AB_product = det_A * det_B

# (b) det(A^{-1}) = 1 / det(A)
det_Ainv = np.linalg.det(np.linalg.inv(A))
det_A_recip = 1.0 / det_A

# (c) det(cA) = c^n * det(A)
det_cA = np.linalg.det(c * A)
det_cA_formula = (c ** n) * det_A

# (d) det(A^T) = det(A)
det_AT = np.linalg.det(A.T)

# (e) det(A) = product of eigenvalues
eigvals_A = np.linalg.eigvals(A)
det_from_eig = np.real(np.prod(eigvals_A))

print(f'(a) det(AB) = {det_AB:.6f}')
print(f'    det(A)*det(B) = {det_AB_product:.6f}')
print(f'\n(b) det(A^-1) = {det_Ainv:.6f}')
print(f'    1/det(A) = {det_A_recip:.6f}')
print(f'\n(c) det(3A) = {det_cA:.6f}')
print(f'    3^4 * det(A) = {det_cA_formula:.6f}')
print(f'\n(d) det(A^T) = {det_AT:.6f}')
print(f'    det(A) = {det_A:.6f}')
print(f'\n(e) det(A) = {det_A:.6f}')
print(f'    prod(eigenvalues) = {det_from_eig:.6f}')

In [ ]:
# Verification cell for Exercise 10
assert np.isclose(det_AB, det_AB_product, rtol=1e-10), \
    f'det(AB) != det(A)*det(B): {det_AB:.6f} vs {det_AB_product:.6f}'
assert np.isclose(det_Ainv, det_A_recip, rtol=1e-10), \
    f'det(A^-1) != 1/det(A): {det_Ainv:.6f} vs {det_A_recip:.6f}'
assert np.isclose(det_cA, det_cA_formula, rtol=1e-10), \
    f'det(cA) != c^n det(A): {det_cA:.6f} vs {det_cA_formula:.6f}'
assert np.isclose(det_AT, det_A, rtol=1e-10), \
    f'det(A^T) != det(A): {det_AT:.6f} vs {det_A:.6f}'
assert np.isclose(det_A, det_from_eig, rtol=1e-8), \
    f'det(A) != prod(eigenvalues): {det_A:.6f} vs {det_from_eig:.6f}'
print('Exercise 10 PASSED: All determinant properties verified')
print(f'  (a) det(AB)={det_AB:.4f}, det(A)det(B)={det_AB_product:.4f}')
print(f'  (b) det(A^-1)={det_Ainv:.4f}, 1/det(A)={det_A_recip:.4f}')
print(f'  (c) det(cA)={det_cA:.4f}, c^n det(A)={det_cA_formula:.4f}, c={c}, n={n}')
print(f'  (d) det(A^T)={det_AT:.4f}, det(A)={det_A:.4f}')
print(f'  (e) det(A)={det_A:.4f}, prod(eigenvalues)={det_from_eig:.4f}')

### Exercise 11 (Intermediate): Cofactor Expansion Determinant

The cofactor expansion computes the determinant recursively:
$$\det(A) = \sum_{j=1}^{n} (-1)^{1+j} \, A_{1j} \, \det(M_{1j})$$
where $M_{1j}$ is the $(n-1) \times (n-1)$ minor obtained by deleting row 1 and column $j$.

This runs in $O(n!)$ time -- completely impractical for large matrices, but instructive for understanding what the determinant computes.

In [ ]:
# Exercise 11: Recursive cofactor expansion determinant

def det_cofactor(A):
    """Compute determinant via cofactor expansion (recursive, O(n!))."""
    A = np.array(A, dtype=float)
    n = A.shape[0]
    if n == 1:
        return A[0, 0]
    if n == 2:
        return A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]
    d = 0.0
    for j in range(n):
        minor = np.delete(np.delete(A, 0, axis=0), j, axis=1)
        d += ((-1) ** j) * A[0, j] * det_cofactor(minor)
    return d

# Verify correctness
print('Correctness check:')
for n in [2, 3, 4, 5]:
    M = np.random.randn(n, n)
    det_custom = det_cofactor(M)
    det_numpy = np.linalg.det(M)
    assert np.isclose(det_custom, det_numpy, rtol=1e-8), \
        f'det_cofactor wrong for n={n}: got {det_custom}, expected {det_numpy}'
    print(f'  n={n}: det_cofactor={det_custom:.6f}, np.linalg.det={det_numpy:.6f}')

# Time cofactor expansion for n = 2..10 to show O(n!) growth
print('\nTiming cofactor expansion (watch it explode):')
sizes = list(range(2, 11))
times_cofactor = []

for n in sizes:
    M = np.random.randn(n, n)
    reps = max(1, 1000 // (math.factorial(n) + 1))
    start = time.perf_counter()
    for _ in range(max(reps, 1)):
        det_cofactor(M)
    elapsed = (time.perf_counter() - start) / max(reps, 1)
    times_cofactor.append(elapsed)
    print(f'  n={n:2d}: {elapsed*1e6:.2f} us')

plt.figure(figsize=(8, 4))
plt.semilogy(sizes, times_cofactor, 'bo-', label='Cofactor expansion')
plt.xlabel('Matrix size n')
plt.ylabel('Time (seconds, log scale)')
plt.title('Cofactor Expansion: O(n!) Runtime Growth')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Verification cell for Exercise 11
for n in [2, 3, 4, 5, 6]:
    M = np.random.randn(n, n)
    assert np.isclose(det_cofactor(M), np.linalg.det(M), rtol=1e-8), \
        f'det_cofactor failed for n={n}'
print('Exercise 11 PASSED: Cofactor expansion correct for n=2..6')
print(f'  Time ratios (should grow roughly as (n+1)! / n! = n+1):')
for i in range(1, len(times_cofactor)):
    ratio = times_cofactor[i] / times_cofactor[i-1] if times_cofactor[i-1] > 0 else float('nan')
    print(f'    n={sizes[i-1]}->{sizes[i]}: ratio={ratio:.1f} (expected ~{sizes[i]})')

### Exercise 12 (Challenging): LU-Based Determinant

In practice, nobody uses cofactor expansion. The standard approach is LU decomposition:
$$PA = LU$$
where $P$ is a permutation matrix, $L$ is unit lower-triangular ($\det(L) = 1$), and $U$ is upper-triangular.

Then:
$$\det(A) = \det(P)^{-1} \cdot \det(L) \cdot \det(U) = \text{sign}(P) \cdot \prod_i U_{ii}$$

This runs in $O(n^3)$ -- the same cost as a matrix multiply.

In [ ]:
# Exercise 12: LU-based determinant
from scipy.linalg import lu

def det_lu(A):
    """Compute determinant via LU decomposition (O(n^3))."""
    P, L, U = lu(A)
    # det(L) = 1 (unit lower triangular)
    det_U = np.prod(np.diag(U))
    # det(P) = +1 or -1 (permutation matrix)
    det_P = np.linalg.det(P)
    return det_P * det_U

# Verify correctness on various sizes
print('Correctness check:')
for n in [3, 5, 8, 10]:
    M = np.random.randn(n, n)
    det_lu_val = det_lu(M)
    det_np_val = np.linalg.det(M)
    assert np.isclose(det_lu_val, det_np_val, rtol=1e-8), \
        f'det_lu wrong for n={n}: got {det_lu_val:.6f}, expected {det_np_val:.6f}'
    print(f'  n={n:2d}: det_lu={det_lu_val:.6f}, np.det={det_np_val:.6f}')

# Speed comparison: LU vs cofactor
print('\nSpeed comparison (microseconds):')
for n in [4, 6, 8, 10]:
    M = np.random.randn(n, n)
    reps = 200

    start = time.perf_counter()
    for _ in range(reps):
        det_lu(M)
    t_lu = (time.perf_counter() - start) / reps * 1e6

    if n <= 8:
        start = time.perf_counter()
        for _ in range(reps):
            det_cofactor(M)
        t_cofactor = (time.perf_counter() - start) / reps * 1e6
        print(f'  n={n}: LU={t_lu:.1f}us, Cofactor={t_cofactor:.1f}us, Speedup={t_cofactor/t_lu:.1f}x')
    else:
        print(f'  n={n}: LU={t_lu:.1f}us, Cofactor=too slow')

In [ ]:
# Verification cell for Exercise 12
for n in [3, 5, 8, 12, 15]:
    M = np.random.randn(n, n)
    assert np.isclose(det_lu(M), np.linalg.det(M), rtol=1e-8), \
        f'det_lu failed for n={n}'
print('Exercise 12 PASSED: LU-based determinant correct for n=3,5,8,12,15')

### Exercise 13 (Challenging): Volume Interpretation of the Determinant

The absolute value of the determinant gives the volume of the parallelepiped formed by the columns (or rows) of the matrix.

In 2D: $|\det(A)|$ = area of the parallelogram formed by applying $A$ to the unit square.

In 3D: $|\det(A)|$ = volume of the parallelepiped formed by the column vectors. This equals the scalar triple product $|a_1 \cdot (a_2 \times a_3)|$.

In [ ]:
# Exercise 13: Volume interpretation of determinant

# ---- 2D: Unit square -> parallelogram ----
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Unit square corners (close the polygon)
unit_square = np.array([[0,1,1,0,0],
                         [0,0,1,1,0]], dtype=float)

matrices_2d = [
    ('Identity', np.eye(2)),
    ('Scaling (2,3)', np.array([[2.,0],[0,3.]])),
    ('Rotation 45 deg', np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
                                   [np.sin(np.pi/4),  np.cos(np.pi/4)]])),
    ('Shear', np.array([[1.,2.],[0.,1.]])),
]

for col, (name, A2) in enumerate(matrices_2d):
    transformed = A2 @ unit_square
    det_val = np.linalg.det(A2)
    area = abs(det_val)

    ax_orig = axes[0][col]
    ax_trans = axes[1][col]

    ax_orig.fill(unit_square[0], unit_square[1], alpha=0.3, color='blue')
    ax_orig.plot(unit_square[0], unit_square[1], 'b-')
    ax_orig.set_xlim(-0.5, 3.5)
    ax_orig.set_ylim(-0.5, 3.5)
    ax_orig.set_aspect('equal')
    ax_orig.set_title(f'{name}\nUnit square (area=1)', fontsize=8)
    ax_orig.axhline(0, color='k', lw=0.5)
    ax_orig.axvline(0, color='k', lw=0.5)

    ax_trans.fill(transformed[0], transformed[1], alpha=0.3, color='red')
    ax_trans.plot(transformed[0], transformed[1], 'r-')
    ax_trans.set_xlim(-3.5, 3.5)
    ax_trans.set_ylim(-3.5, 3.5)
    ax_trans.set_aspect('equal')
    ax_trans.set_title(f'Transformed\ndet={det_val:.2f}, area={area:.2f}', fontsize=8)
    ax_trans.axhline(0, color='k', lw=0.5)
    ax_trans.axvline(0, color='k', lw=0.5)

axes[0][0].set_ylabel('Original', fontsize=10)
axes[1][0].set_ylabel('Transformed', fontsize=10)
plt.suptitle('2D: |det(A)| = Area of Parallelogram', fontsize=12)
plt.tight_layout()
plt.show()

# ---- 3D: Unit cube -> parallelepiped ----
print('3D Parallelepiped Volumes (|det| vs scalar triple product):')
matrices_3d = [
    ('Identity', np.eye(3)),
    ('Scaling(2,3,4)', np.diag([2., 3., 4.])),
    ('Triangular', np.array([[1.,2.,0.],[0.,1.,1.],[0.,0.,2.]])),
]

for name, A3 in matrices_3d:
    det_3d = np.linalg.det(A3)
    a1, a2, a3 = A3[:, 0], A3[:, 1], A3[:, 2]
    vol_triple = abs(np.dot(a1, np.cross(a2, a3)))
    print(f'  {name}: |det|={abs(det_3d):.4f}, triple product vol={vol_triple:.4f}')
    assert np.isclose(abs(det_3d), vol_triple, atol=1e-10), \
        f'Volume mismatch for {name}'

In [ ]:
# Verification cell for Exercise 13
# Re-verify area = |det| for all 2D matrices
for name, A2 in matrices_2d:
    det_val = np.linalg.det(A2)
    col1 = A2[:, 0]
    col2 = A2[:, 1]
    area_cross = abs(col1[0]*col2[1] - col1[1]*col2[0])  # 2D cross product
    assert np.isclose(abs(det_val), area_cross, atol=1e-10), \
        f'{name}: |det|={abs(det_val):.4f} != area={area_cross:.4f}'

# Re-verify 3D volumes
for name, A3 in matrices_3d:
    det_3d = np.linalg.det(A3)
    a1, a2, a3 = A3[:, 0], A3[:, 1], A3[:, 2]
    vol = abs(np.dot(a1, np.cross(a2, a3)))
    assert np.isclose(abs(det_3d), vol, atol=1e-10), \
        f'{name}: |det|={abs(det_3d):.4f} != triple product={vol:.4f}'

print('Exercise 13 PASSED')
print('  2D: |det(A)| = area of parallelogram verified for all test matrices')
print('  3D: |det(A)| = volume of parallelepiped verified for all test matrices')

---

## Summary

### Section 2.8: SVD

- **Every** real matrix $A \in \mathbb{R}^{m \times n}$ has an SVD: $A = U \Sigma V^\top$.
- Geometric meaning: any linear transformation = rotate ($V^\top$) $\to$ scale ($\Sigma$) $\to$ rotate ($U$).
- Singular values $\sigma_i = \sqrt{\lambda_i(A^\top A)}$; right singular vectors = eigenvectors of $A^\top A$.
- **Eckart-Young:** Best rank-$k$ approximation is the truncated SVD. Error $= \sqrt{\sum_{i>k} \sigma_i^2}$.
- **ML connections:** PCA, LSA, recommender systems, image compression.

### Section 2.9: Moore-Penrose Pseudoinverse

- $A^+ = V \Sigma^+ U^\top$: reciprocate non-zero singular values.
- Overdetermined ($m > n$): $A^+ b$ = least-squares solution (minimizes $\|Ax - b\|^2$).
- Underdetermined ($m < n$): $A^+ b$ = minimum-norm solution (smallest $\|x\|$).
- **ML connection:** Linear regression's closed-form solution $\hat{\theta} = X^+ y$.

### Section 2.10: Trace

- $\text{tr}(A) = \sum_i A_{ii} = \sum_i \lambda_i$.
- **Cyclic invariance:** $\text{tr}(ABC) = \text{tr}(CAB) = \text{tr}(BCA)$.
- **Frobenius norm:** $\|A\|_F^2 = \text{tr}(A^\top A)$.
- **Trace trick:** $\frac{\partial}{\partial A} \text{tr}(AB) = B^\top$ -- used constantly in matrix calculus.

### Section 2.11: Determinant

- $\det(A) = \prod_i \lambda_i$ -- product of eigenvalues.
- $|\det(A)|$ = volume scaling factor. $\det = 0$ means a dimension collapses. Sign = orientation.
- Key properties: $\det(AB) = \det(A)\det(B)$, $\det(A^{-1}) = 1/\det(A)$, $\det(cA) = c^n \det(A)$.
- **ML connections:** Multivariate Gaussian normalizing constant ($\det(\Sigma)$), normalizing flows (Jacobian determinant), GMM log-likelihoods.

### The Big Picture

| Concept | Eigenvalue connection | What it measures |
|---------|----------------------|------------------|
| Trace | $\sum_i \lambda_i$ | "Average" scaling |
| Determinant | $\prod_i \lambda_i$ | Volume scaling |
| Singular values | $\sqrt{\lambda_i(A^\top A)}$ | Axis-by-axis stretching |
| Frobenius norm | $\sqrt{\sum_i \sigma_i^2}$ | Total "energy" of the transformation |